In [1]:
import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from ucimlrepo import fetch_ucirepo
import copy
from sklearn.model_selection import TimeSeriesSplit

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
df = pd.read_csv("data/day.csv")

df = df.sort_values("dteday").reset_index(drop=True)
print(df.shape)

y = df["cnt"].astype(float).values
X = df.drop(columns=["instant", "dteday", "casual", "registered", "cnt"])
X = pd.get_dummies(X, columns=["season", "mnth", "weekday", "weathersit"], dtype=float)

n = len(X)
i_tr, i_va = int(0.70 * n), int(0.85 * n)

y_mean, y_std = y[:i_tr].mean(), y[:i_tr].std()
y_s = (y - y_mean) / y_std

def to_t(a): return torch.tensor(np.asarray(a), dtype=torch.float32)

Xtr, Xva, Xte = to_t(X.values[:i_tr]), to_t(X.values[i_tr:i_va]), to_t(X.values[i_va:])
ytr, yva, yte = to_t(y_s[:i_tr]).unsqueeze(1), to_t(y_s[i_tr:i_va]).unsqueeze(1), to_t(y_s[i_va:]).unsqueeze(1)

n_features = Xtr.shape[1]
print(n_features, "features |", len(Xtr), "/", len(Xva), "/", len(Xte))

(731, 16)
33 features | 511 / 110 / 110


In [3]:
def build_model(hidden=(), dropout=0.0):
    layers, d = [], n_features
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        d = h
    layers.append(nn.Linear(d, 1))
    return nn.Sequential(*layers)

def make_name(hidden, dropout, weight_decay, patience, tag=""):
    parts = ["linear" if not hidden else "mlp_" + "x".join(map(str, hidden))]
    if dropout:      parts.append(f"do{dropout}")
    if weight_decay: parts.append(f"wd{weight_decay}")
    if patience:     parts.append(f"es{patience}")
    return "_".join(parts) + tag

def train_model(model, Xtr_, ytr_, Xva_, yva_, lr, epochs, batch_size,
                weight_decay, patience, writer=None, prefix=""):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(Xtr_, ytr_), batch_size=batch_size, shuffle=True)
    Xtr_d, ytr_d, Xva_d, yva_d = Xtr_.to(device), ytr_.to(device), Xva_.to(device), yva_.to(device)

    best_val, best_state, best_epoch, wait = float("inf"), None, 0, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            train_loss = loss_fn(model(Xtr_d), ytr_d).item()
            val_loss = loss_fn(model(Xva_d), yva_d).item()

        if writer:
            writer.add_scalars(f"{prefix}Loss", {"train": train_loss, "val": val_loss}, epoch)
            writer.add_scalar(f"{prefix}RMSE_val_velos", (val_loss ** 0.5) * y_std, epoch)
            if not prefix and epoch % 10 == 0:
                for pname, p in model.named_parameters():
                    writer.add_histogram(pname, p, epoch)

        if val_loss < best_val - 1e-6:
            best_val, best_epoch, wait = val_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if patience and wait >= patience:
                break

    if patience and best_state is not None:
        model.load_state_dict(best_state)
        final_val = best_val
    else:
        final_val = val_loss
    return {"val_loss": final_val, "best_epoch": best_epoch, "last_epoch": epoch}

def run_experiment(hidden=(), lr=1e-3, epochs=300, batch_size=32,
                   dropout=0.0, weight_decay=0.0, patience=None, tag=""):
    name = make_name(hidden, dropout, weight_decay, patience, tag)
    writer = SummaryWriter(f"runs/bike/{name}")
    model = build_model(hidden, dropout).to(device)

    model.eval()
    writer.add_graph(model, Xtr[:1].to(device))
    writer.add_text("config", f"hidden={hidden}, lr={lr}, epochs={epochs}, bs={batch_size}, "
                              f"dropout={dropout}, wd={weight_decay}, patience={patience}, "
                              f"params={sum(p.numel() for p in model.parameters())}")

    res = train_model(model, Xtr, ytr, Xva, yva, lr, epochs, batch_size,
                      weight_decay, patience, writer)

    model.eval()
    with torch.no_grad():
        test_rmse = (nn.MSELoss()(model(Xte.to(device)), yte.to(device)).item() ** 0.5) * y_std
    val_rmse = (res["val_loss"] ** 0.5) * y_std
    writer.add_scalar("RMSE_test_velos", test_rmse, res["last_epoch"])
    writer.close()
    print(f"{name:35s} | val RMSE = {val_rmse:6.1f} | test RMSE = {test_rmse:6.1f} "
          f"| meilleure epoch = {res['best_epoch']}/{res['last_epoch']}")
    return model

def cross_validate(hidden=(), n_splits=5, lr=1e-3, epochs=300, batch_size=32,
                   dropout=0.0, weight_decay=0.0, patience=None, tag=""):
    """Validation croisée chronologique sur train+val. Le test reste intact."""
    name = make_name(hidden, dropout, weight_decay, patience, tag) + "_cv"
    writer = SummaryWriter(f"runs/bike/{name}")
    X_all, y_all = torch.cat([Xtr, Xva]), torch.cat([ytr, yva])

    rmses = []
    for k, (tr_idx, va_idx) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(X_all)):
        model = build_model(hidden, dropout).to(device)
        res = train_model(model, X_all[tr_idx], y_all[tr_idx], X_all[va_idx], y_all[va_idx],
                          lr, epochs, batch_size, weight_decay, patience,
                          writer, prefix=f"fold{k}/")
        rmses.append((res["val_loss"] ** 0.5) * y_std)

    writer.add_scalar("CV/RMSE_mean", np.mean(rmses), 0)
    writer.add_scalar("CV/RMSE_std", np.std(rmses), 0)
    writer.close()
    print(f"{name:35s} | CV RMSE = {np.mean(rmses):6.1f} ± {np.std(rmses):.1f} vélos/jour "
          f"(folds : {[round(r) for r in rmses]})")

In [7]:
%load_ext tensorboard
%tensorboard --logdir runs/bike

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 8504), started 1:13:20 ago. (Use '!kill 8504' to kill it.)

In [5]:
run_experiment(hidden=(), lr=1e-3, epochs=300, batch_size=32, tag="1_node")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


linear1_node: val RMSE = 859.2 | test RMSE = 1232.9 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=1, bias=True)
)

In [6]:
run_experiment(hidden=(2,), lr=1e-3, epochs=300, batch_size=32, tag="2_node")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_22_node: val RMSE = 981.3 | test RMSE = 1069.2 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=2, bias=True)
  (1): ReLU()
  (2): Linear(in_features=2, out_features=1, bias=True)
)

In [8]:
run_experiment(hidden=(4,), lr=1e-3, epochs=300, batch_size=32, tag="4_node")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_44_node: val RMSE = 909.0 | test RMSE = 1227.0 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=1, bias=True)
)

In [9]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, tag="_run_3")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_run_3: val RMSE = 845.3 | test RMSE = 1109.7 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [10]:
run_experiment(hidden=(8,4), lr=1e-3, epochs=200, batch_size=32, tag="_run_4")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_8x4_run_4: val RMSE = 917.9 | test RMSE = 1275.8 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

In [11]:
run_experiment(hidden=(16,8), lr=1e-3, epochs=200, batch_size=32, tag="_run_5")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_16x8_run_5: val RMSE = 973.6 | test RMSE = 1271.7 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
)

In [12]:
run_experiment(hidden=(4,4), lr=1e-3, epochs=200, batch_size=32, tag="_run_6")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x4_run_6: val RMSE = 1131.9 | test RMSE = 1353.3 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

In [13]:
run_experiment(hidden=(4,2), lr=1e-4, epochs=200, batch_size=32, tag="_run_7_lowerlr")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_run_7_lowerlr: val RMSE = 1750.7 | test RMSE = 1933.7 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [18]:
run_experiment(hidden=(4,2), lr=1e-4, epochs=900, batch_size=32, tag="_run_8_lowerlr")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_run_8_lowerlr: val RMSE = 884.4 | test RMSE = 1262.6 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [17]:
run_experiment(hidden=(4,2), lr=1e-4, epochs=2000, batch_size=32, tag="_run_9_lowerlr")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_run_9_lowerlr: val RMSE = 918.8 | test RMSE = 1307.7 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [19]:
run_experiment(hidden=(4,3), lr=1e-3, epochs=200, batch_size=32, tag="_run_10")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x3_run_10_: val RMSE = 1226.7 | test RMSE = 1355.0 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=3, bias=True)
  (3): ReLU()
  (4): Linear(in_features=3, out_features=1, bias=True)
)

In [20]:
run_experiment(hidden=(4,3), lr=1e-3, epochs=600, batch_size=32, tag="_run_11_more_epoch")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x3_run_10_more_epoch: val RMSE = 1373.4 | test RMSE = 1506.3 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=3, bias=True)
  (3): ReLU()
  (4): Linear(in_features=3, out_features=1, bias=True)
)

In [21]:
run_experiment(hidden=(4,3), lr=1e-3, epochs=600, batch_size=16, tag="_run_12_less_batch")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x3_run_11_batch: val RMSE = 1125.7 | test RMSE = 1229.8 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=3, bias=True)
  (3): ReLU()
  (4): Linear(in_features=3, out_features=1, bias=True)
)

In [22]:
run_experiment(hidden=(4,3), lr=1e-3, epochs=600, batch_size=64, tag="_run_13_more_batch")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x3_run_12_batch: val RMSE = 1011.8 | test RMSE = 1295.3 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=3, bias=True)
  (3): ReLU()
  (4): Linear(in_features=3, out_features=1, bias=True)
)

In [23]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=600, batch_size=64, tag="_run_14_more_batch")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_run_13_batch: val RMSE = 1392.2 | test RMSE = 1526.4 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [24]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=600, batch_size=16, tag="_run_15_less_batch")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_run_14_batch: val RMSE = 2505.9 | test RMSE = 1603.6 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [25]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=85, batch_size=32, tag="_run_16_less_epoch")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_run_15: val RMSE = 1694.2 | test RMSE = 1730.9 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [26]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=100, batch_size=32, tag="_run_17_less_epoch")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_run_16: val RMSE = 924.0 | test RMSE = 1332.0 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [27]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=130, batch_size=32, tag="_run_18_less_epoch")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_run_17: val RMSE = 893.9 | test RMSE = 1328.6 vélos/h


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [32]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0.01, weight_decay=0.0, patience=None, tag="_run_19_dropout")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_do0.01_run_18_dropout       | val RMSE = 1027.1 | test RMSE = 1295.6 | meilleure epoch = 57/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.01, inplace=False)
  (3): Linear(in_features=4, out_features=2, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.01, inplace=False)
  (6): Linear(in_features=2, out_features=1, bias=True)
)

In [33]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0.03, weight_decay=0.0, patience=None, tag="_run_20_dropout")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_do0.03_run_19_dropout       | val RMSE = 3082.4 | test RMSE = 2663.7 | meilleure epoch = 0/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.03, inplace=False)
  (3): Linear(in_features=4, out_features=2, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.03, inplace=False)
  (6): Linear(in_features=2, out_features=1, bias=True)
)

In [34]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0.3, weight_decay=0.0, patience=None, tag="_run_21_dropout")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_do0.3_run_20_dropout        | val RMSE = 2016.2 | test RMSE = 1913.4 | meilleure epoch = 163/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=4, out_features=2, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=2, out_features=1, bias=True)
)

In [36]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0.1, weight_decay=0.0, patience=None, tag="_run_22_dropout")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_do0.1_run_21_dropout        | val RMSE = 1482.3 | test RMSE = 1446.5 | meilleure epoch = 199/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=4, out_features=2, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.1, inplace=False)
  (6): Linear(in_features=2, out_features=1, bias=True)
)

In [37]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0.01, patience=None, tag="_run_23_decay")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_wd0.01_run_22_decay         | val RMSE =  876.3 | test RMSE = 1308.4 | meilleure epoch = 73/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [38]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0.001, patience=None, tag="_run_24_decay")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_wd0.001_run_22_decay        | val RMSE =  946.1 | test RMSE = 1277.7 | meilleure epoch = 66/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [39]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0.1, patience=None, tag="_run_25_decay")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_wd0.1_run_22_decay          | val RMSE =  997.2 | test RMSE = 1379.5 | meilleure epoch = 198/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [40]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0.05, patience=None, tag="_run_26_decay")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_wd0.05_run_22_decay         | val RMSE =  933.0 | test RMSE = 1324.1 | meilleure epoch = 165/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [41]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0.008, patience=None, tag="_run_28_decay")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_wd0.008_run_22_decay        | val RMSE = 1289.3 | test RMSE = 1381.7 | meilleure epoch = 197/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [42]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0, patience=5, tag="_run_29_patience")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_es5_run_23_patience         | val RMSE = 1416.2 | test RMSE = 1544.3 | meilleure epoch = 199/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [46]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0, patience=100, tag="_run_30_patience")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_es100_run_23_patience       | val RMSE =  820.7 | test RMSE = 1292.8 | meilleure epoch = 56/156


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [47]:
cross_validate(hidden=(4,2), n_splits=5, lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0, patience=0, tag="_run_31_crossval")

mlp_4x2_run_24_crossval_cv          | CV RMSE = 1709.7 ± 786.9 vélos/jour (folds : [1978, 973, 966, 1545, 3088])


In [49]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0.01, patience=100, tag="_run_32_multiparam")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_wd0.01_es100_run_25_multiparam | val RMSE = 1476.8 | test RMSE = 1593.0 | meilleure epoch = 198/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [50]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=600, batch_size=32, dropout=0, weight_decay=0.01, patience=100, tag="_run_33_multiparam")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_wd0.01_es100_run_25_multiparam | val RMSE = 1021.4 | test RMSE = 1212.0 | meilleure epoch = 599/599


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=1, bias=True)
)

In [54]:
run_experiment(hidden=(4,2), lr=1e-3, epochs=200, batch_size=32, dropout=0.01, weight_decay=0.01, patience=None, tag="_run_34_multiparam")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_4x2_do0.005_wd0.01_run_25_multiparam | val RMSE = 1559.6 | test RMSE = 1633.2 | meilleure epoch = 199/199


Sequential(
  (0): Linear(in_features=33, out_features=4, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.005, inplace=False)
  (3): Linear(in_features=4, out_features=2, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.005, inplace=False)
  (6): Linear(in_features=2, out_features=1, bias=True)
)

In [56]:
cross_validate(hidden=(4,2), n_splits=5, lr=1e-3, epochs=200, batch_size=32, dropout=0, weight_decay=0.01, patience=None, tag="_run_35_crossmultiparam")

mlp_4x2_wd0.01_run_26_crossmultiparam_cv | CV RMSE = 1309.2 ± 520.3 vélos/jour (folds : [2200, 695, 1414, 1341, 896])


In [8]:
run_experiment(hidden=(8,4,4), lr=1e-3, epochs=200, batch_size=32, tag="_run_36_complex_model")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_8x4x4_run_27_complex_model      | val RMSE =  960.0 | test RMSE = 1135.7 | meilleure epoch = 87/199


Sequential(
  (0): Linear(in_features=33, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=4, bias=True)
  (5): ReLU()
  (6): Linear(in_features=4, out_features=1, bias=True)
)

In [9]:
run_experiment(hidden=(8,4,4,4), lr=1e-3, epochs=200, batch_size=32, tag="_run_37_complex_model")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_8x4x4x4_run_28_complex_model    | val RMSE = 1422.6 | test RMSE = 1595.6 | meilleure epoch = 179/199


Sequential(
  (0): Linear(in_features=33, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=4, bias=True)
  (5): ReLU()
  (6): Linear(in_features=4, out_features=4, bias=True)
  (7): ReLU()
  (8): Linear(in_features=4, out_features=1, bias=True)
)

In [10]:
run_experiment(hidden=(8,2,2,4), lr=1e-3, epochs=200, batch_size=32, tag="_run_38_complex_model")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_8x2x2x4_run_28_complex_model    | val RMSE = 1094.1 | test RMSE = 1312.9 | meilleure epoch = 38/199


Sequential(
  (0): Linear(in_features=33, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=2, bias=True)
  (3): ReLU()
  (4): Linear(in_features=2, out_features=2, bias=True)
  (5): ReLU()
  (6): Linear(in_features=2, out_features=4, bias=True)
  (7): ReLU()
  (8): Linear(in_features=4, out_features=1, bias=True)
)

In [11]:
run_experiment(hidden=(8,3,4), lr=1e-3, epochs=200, batch_size=32, tag="_run_39_complex_model")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_8x3x4_run_28_complex_model      | val RMSE = 1381.5 | test RMSE = 1489.6 | meilleure epoch = 193/199


Sequential(
  (0): Linear(in_features=33, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=3, bias=True)
  (3): ReLU()
  (4): Linear(in_features=3, out_features=4, bias=True)
  (5): ReLU()
  (6): Linear(in_features=4, out_features=1, bias=True)
)

In [12]:
run_experiment(hidden=(8,8,4), lr=1e-3, epochs=200, batch_size=32, tag="_run_40_complex_model")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_8x8x4_run_28_complex_model      | val RMSE = 1020.6 | test RMSE = 1014.2 | meilleure epoch = 22/199


Sequential(
  (0): Linear(in_features=33, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=4, bias=True)
  (5): ReLU()
  (6): Linear(in_features=4, out_features=1, bias=True)
)

In [13]:
run_experiment(hidden=(8,12,4), lr=1e-3, epochs=200, batch_size=32, tag="_run_41_complex_model")

C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\loicq\Documents\2026_2027\TP_Perceptron\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


mlp_8x12x4_run_28_complex_model     | val RMSE = 1442.6 | test RMSE = 1557.3 | meilleure epoch = 197/199


Sequential(
  (0): Linear(in_features=33, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=12, bias=True)
  (3): ReLU()
  (4): Linear(in_features=12, out_features=4, bias=True)
  (5): ReLU()
  (6): Linear(in_features=4, out_features=1, bias=True)
)

In [ ]:
#TODO Optimizer, test avec models complexes 